# Reconocimiento de llantas de vehículos por keypoints con YOLO y Detectron2
**IAM-811 Machine Learning y Aprendizaje Profundo** · Ing. Alvaro Pinto Mamani

Los modelos de keypoints preentrenados (COCO) solo conocen el esqueleto humano: **no existe un modelo listo para llantas**. Este notebook resuelve el problema en tres etapas:

| Etapa | Método | Entrenamiento |
|---|---|---|
| 1. Zero-shot | YOLOE-seg (prompts *wheel*, *tire*) → máscara de la llanta → 5 keypoints geométricos | No |
| 2. Auto-etiquetado | El método zero-shot genera un dataset en formato YOLO-pose (corregible en Roboflow) | No |
| 3. Modelos de keypoints | **YOLO-pose** y **Detectron2 Keypoint R-CNN** entrenados con ese dataset | Sí |

**Definición de keypoints (por cada llanta):**

| # | Nombre | Significado |
|---|---|---|
| 0 | centro | centro del aro |
| 1 | contacto | punto de contacto con el suelo |
| 2 | superior | punto más alto de la llanta |
| 3 | izquierda | extremo izquierdo |
| 4 | derecha | extremo derecho |

Las llantas se asocian a cada vehículo (auto, bus, minibús, camión) para contar **llantas visibles** y estimar el **número de ejes**, dato útil para el control de peaje. La estimación de ejes supone vista lateral del vehículo.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "Sin GPU: active la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno"
!pip -q install -U ultralytics
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'
# Descomenta la siguiente línea solo si también vas a ejecutar la sección 8.4 (Detectron2):
# !pip -q install --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'

GPU 0: Tesla T4 (UUID: GPU-5a285bf9-36c8-0e08-d75b-3619448ee58a)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Configuración y utilidades

In [ ]:
import os, glob, time, random, shutil, json, cv2, numpy as np, torch, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO, YOLOE
from google.colab import files

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)

try:
    from detectron2 import model_zoo
    from detectron2.config import get_cfg
    from detectron2.engine import DefaultPredictor, DefaultTrainer
    from detectron2.data import MetadataCatalog, DatasetCatalog
    from detectron2.data.datasets import register_coco_instances
    HAY_D2 = True
except Exception as e:
    HAY_D2 = False
    print('Detectron2 no disponible, solo se usarán los métodos YOLO:', e)

# Vehículos (prompts de texto para YOLOE -> clase interna)
PROMPTS_VEHICULO = {'car': 'auto', 'sedan': 'auto', 'bus': 'bus',
                    'minibus': 'minibus', 'van': 'minibus', 'truck': 'camion'}
PROMPTS_LLANTA = ['wheel', 'tire', 'car wheel']
CLASES = {'auto': 'Auto particular', 'bus': 'Bus', 'minibus': 'Minibús', 'camion': 'Camión'}
ETIQUETA_IMG = {'auto': 'Auto', 'bus': 'Bus', 'minibus': 'Minibus', 'camion': 'Camion'}   # sin tildes para OpenCV
COLORES = {'auto': (52, 152, 219), 'bus': (231, 76, 60), 'minibus': (46, 204, 113), 'camion': (243, 156, 18)}

# Keypoints de la llanta
KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
KPT_COLORES = [(255, 255, 255), (255, 0, 0), (0, 200, 255), (0, 255, 0), (255, 0, 255)]   # RGB
ESQUELETO = [(0, 1), (0, 2), (0, 3), (0, 4)]
FLIP_IDX = [0, 1, 2, 4, 3]          # al voltear horizontalmente se intercambian izquierda y derecha
UMBRAL_KP = 0.05                     # confianza mínima para dibujar un keypoint

# Rutas de modelos propios
PESO_YOLOE = 'yoloe-11l-seg.pt'
RUTA_YOLOPOSE = '/content/runs_pose/m3_ejes/weights/best.pt'
DIR_D2 = '/content/runs/d2_llantas'
RUTA_D2 = DIR_D2 + '/model_final.pth'

_cache = {}
def cargar(clave, fabrica):
    if clave not in _cache:
        _cache[clave] = fabrica()
    return _cache[clave]

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0

def nms_simple(items, umbral=0.6):
    # Elimina duplicados entre prompts (p. ej. 'wheel' y 'tire' sobre la misma llanta)
    items = sorted(items, key=lambda d: -d['conf'])
    res = []
    for d in items:
        if all(iou(d['box'], r['box']) < umbral for r in res):
            res.append(d)
    return res

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dispositivo: cuda
Detectron2 no disponible, solo se usarán los métodos YOLO: No module named 'detectron2'


## 3. Subir imagen

In [ ]:
subidos = files.upload()
IMAGENES = list(subidos.keys())
if not IMAGENES:
    !wget -q -O ejemplo.jpg https://ultralytics.com/images/bus.jpg
    IMAGENES = ['ejemplo.jpg']
IMG_PATH = IMAGENES[0]
img_bgr = cv2.imread(IMG_PATH)
print('Imagen activa:', IMG_PATH, img_bgr.shape)
plt.figure(figsize=(8, 6)); plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## 4. Detección de vehículos y llantas (YOLOE) y keypoints geométricos

In [ ]:
def modelo_yoloe():
    nombres = list(PROMPTS_VEHICULO) + PROMPTS_LLANTA
    def crear():
        m = YOLOE(PESO_YOLOE)
        m.set_classes(nombres, m.get_text_pe(nombres))
        return m
    return cargar(PESO_YOLOE + str(tuple(nombres)), crear)

IMGSZ = 1024          # resolución de inferencia; ayuda con llantas pequeñas e imágenes de baja resolución
CONF_LLANTA = 0.15    # YOLOE da puntajes bajos a objetos pequeños: las llantas usan su propio umbral
DEPURAR = False       # True: imprime todas las detecciones crudas de YOLOE

def analizar_yoloe(img_bgr, conf=0.25):
    # Devuelve vehículos [{clase, conf, box}] y llantas segmentadas [{conf, box, mask}]
    r = modelo_yoloe().predict(img_bgr, conf=min(conf, CONF_LLANTA), imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    H, W = r.orig_shape
    vehiculos, llantas = [], []
    if DEPURAR:
        crudas = [(r.names[int(c)], round(float(s), 3)) for c, s in zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy())]
        print('Detecciones crudas de YOLOE (prompt, confianza):', crudas if crudas else 'ninguna')
    if len(r.boxes) == 0:
        return vehiculos, llantas
    clases = r.boxes.cls.cpu().numpy().astype(int)
    confs = r.boxes.conf.cpu().numpy()
    cajas = r.boxes.xyxy.cpu().numpy()
    polis = r.masks.xy if r.masks is not None else [np.zeros((0, 2))] * len(clases)
    for c, s, b, poly in zip(clases, confs, cajas, polis):
        nombre = r.names[c]
        if nombre in PROMPTS_VEHICULO and s >= conf:
            vehiculos.append({'clase': PROMPTS_VEHICULO[nombre], 'conf': float(s), 'box': b.tolist()})
        elif nombre in PROMPTS_LLANTA and s >= CONF_LLANTA and len(poly) >= 3:
            m = np.zeros((H, W), np.uint8)
            cv2.fillPoly(m, [poly.astype(np.int32)], 1)
            llantas.append({'conf': float(s), 'box': b.tolist(), 'mask': m})
    return nms_simple(vehiculos, 0.7), nms_simple(llantas, 0.5)

def kps_de_mascara(mask):
    # 5 keypoints sub-píxel a partir de la elipse ajustada al contorno de la llanta
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 12:
        return None
    pts = c.reshape(-1, 2).astype(float)
    if len(pts) >= 5:
        (cx, cy), (MA, ma), angle = cv2.fitEllipse(c)
        theta = np.deg2rad(angle)
        a, b = MA / 2.0, ma / 2.0
        t = np.linspace(0, 2 * np.pi, 360, endpoint=False)
        ex = cx + a * np.cos(t) * np.cos(theta) - b * np.sin(t) * np.sin(theta)
        ey = cy + a * np.cos(t) * np.sin(theta) + b * np.sin(t) * np.cos(theta)
        contacto = [ex[np.argmax(ey)], np.max(ey)]
        superior = [ex[np.argmin(ey)], np.min(ey)]
        izquierda = [np.min(ex), ey[np.argmin(ex)]]
        derecha = [np.max(ex), ey[np.argmax(ex)]]
        k = np.array([[cx, cy], contacto, superior, izquierda, derecha])
    else:
        cx, cy = pts.mean(0)
        k = np.array([
            [cx, cy],
            pts[pts[:, 1].argmax()],
            pts[pts[:, 1].argmin()],
            pts[pts[:, 0].argmin()],
            pts[pts[:, 0].argmax()],
        ])
    return np.concatenate([k, np.ones((5, 1))], axis=1)

In [ ]:
# Precarga de YOLOE (la primera vez descarga los pesos y el codificador de texto: 1 a 3 minutos)
t0 = time.time()
modelo_yoloe()
print('YOLOE listo en ' + format(time.time() - t0, '.1f') + ' s')

/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


YOLOE listo en 23.9 s


## 5. Métodos de keypoints
Todos devuelven `vehiculos` y `llantas = [{conf, box, kps (5×3)}]`. Los métodos entrenados aparecen cuando existen sus pesos (sección 8).

In [ ]:
def kp_zeroshot(img_bgr, conf=0.3):
    vehiculos, segs = analizar_yoloe(img_bgr, conf)
    llantas = []
    for s in segs:
        k = kps_de_mascara(s['mask'])
        if k is not None:
            k[:, 2] = s['conf']
            llantas.append({'conf': s['conf'], 'box': s['box'], 'kps': k})
    return vehiculos, llantas

def kp_yolopose(img_bgr, conf=0.3):
    m = cargar(RUTA_YOLOPOSE, lambda: YOLO(RUTA_YOLOPOSE))
    r = m.predict(img_bgr, conf=conf, device=DEVICE, verbose=False)[0]
    llantas = []
    if r.keypoints is not None and len(r.boxes):
        xy = r.keypoints.xy.cpu().numpy()
        kc = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(xy.shape[:2])
        for b, s, p, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), xy, kc):
            llantas.append({'conf': float(s), 'box': b.tolist(), 'kps': np.concatenate([p, c[:, None]], 1)})
    vehiculos, _ = analizar_yoloe(img_bgr, conf)
    return vehiculos, llantas

def construir_cfg_d2():
    yaml = 'COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml'
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(yaml))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(yaml)
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = len(KPT_NOMBRES)
    cfg.TEST.KEYPOINT_OKS_SIGMAS = [0.05] * len(KPT_NOMBRES)
    cfg.MODEL.DEVICE = DEVICE
    cfg.OUTPUT_DIR = DIR_D2
    return cfg

def kp_detectron(img_bgr, conf=0.5):
    def crear():
        cfg = construir_cfg_d2()
        cfg.MODEL.WEIGHTS = RUTA_D2
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.05
        return DefaultPredictor(cfg)
    inst = cargar(RUTA_D2, crear)(img_bgr)['instances'].to('cpu')
    llantas = []
    for b, s, k in zip(inst.pred_boxes.tensor.numpy(), inst.scores.numpy(), inst.pred_keypoints.numpy()):
        if s >= conf:
            llantas.append({'conf': float(s), 'box': b.tolist(), 'kps': k})
    vehiculos, _ = analizar_yoloe(img_bgr, conf)
    return vehiculos, llantas

def metodos_disponibles():
    met = {'Zero-shot · YOLOE-seg + keypoints geométricos': kp_zeroshot}
    if os.path.exists(RUTA_YOLOPOSE):
        met['YOLO-pose entrenado'] = kp_yolopose
    if HAY_D2 and os.path.exists(RUTA_D2):
        met['Detectron2 Keypoint R-CNN entrenado'] = kp_detectron
    return met

## 6. Asociación llanta → vehículo, conteo de ejes y visualización
Los ejes se cuentan solo con las llantas del **lado cercano** a la cámara. Las del lado lejano (parcialmente tapadas por la carrocería) y los aros detectados como objeto aparte se descartan. En vista frontal o posterior los ejes no son determinables.

In [ ]:
UMBRAL_CONTENCION = 0.7    # descarta aros detectados dentro de la llanta
UMBRAL_CERCANA = 0.58      # altura visible mínima para considerar llanta del lado cercano
SEPARACION_EJE = 0.52      # distancia mínima en x entre ejes (en diámetros de llanta)

def altura_visible(l):
    return max(1.0, l['kps'][1, 1] - l['kps'][2, 1])          # contacto.y - superior.y

def contenida(a, b):
    # fracción de la caja a que cae dentro de la caja b
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area = max(1e-6, (a[2] - a[0]) * (a[3] - a[1]))
    return inter / area

def limpiar_llantas(llantas):
    orden = sorted(llantas, key=lambda l: -(l['box'][2] - l['box'][0]) * (l['box'][3] - l['box'][1]))
    res = []
    for l in orden:
        if all(contenida(l['box'], r['box']) < UMBRAL_CONTENCION for r in res):
            res.append(l)
    return res

def analizar_ejes(v, ls):
    # Devuelve (ejes, vista, nota). ejes = None si no es determinable.
    if not ls:
        return None, '-', 'Sin llantas detectadas'
    hmax = max(altura_visible(l) for l in ls)
    cercanas = [l for l in ls if altura_visible(l) >= UMBRAL_CERCANA * hmax]
    x1, y1, x2, y2 = v['box']
    proporcion = (x2 - x1) / max(1.0, y2 - y1)
    # Vista frontal/posterior: 2 llantas angostas (elipses de canto) a la misma altura y caja poco alargada
    if len(cercanas) == 2 and proporcion < 1.6:
        a, b = cercanas
        angostas = all((l['kps'][4, 0] - l['kps'][3, 0]) / altura_visible(l) < 0.65 for l in cercanas)
        misma_altura = abs(a['kps'][1, 1] - b['kps'][1, 1]) < 0.15 * hmax
        if angostas and misma_altura:
            return None, 'frontal/posterior', 'No determinable en esta vista'
    xs = sorted(l['kps'][0, 0] for l in cercanas)
    diam = float(np.median([altura_visible(l) for l in cercanas]))
    ejes = 1 + sum(1 for p, q in zip(xs, xs[1:]) if q - p > SEPARACION_EJE * diam)
    redondez = np.median([(l['kps'][4, 0] - l['kps'][3, 0]) / altura_visible(l) for l in cercanas])
    vista = 'lateral' if redondez > 0.8 else '3/4'
    nota = ''
    if ejes < 2:
        ejes, nota = 2, 'Mínimo asumido (solo se detectó 1 eje)'
    return ejes, vista, nota

def asociar(vehiculos, llantas):
    llantas = limpiar_llantas(llantas)
    for v in vehiculos:
        v['llantas'] = []
    for i, l in enumerate(llantas):
        cx, cy = l['kps'][0, :2]
        l['vehiculo'] = None
        mejor, area_min = None, None
        for j, v in enumerate(vehiculos):
            x1, y1, x2, y2 = v['box']
            w, h = x2 - x1, y2 - y1
            # el centro de la llanta debe estar dentro del vehículo y en su parte baja
            if x1 - 0.05 * w <= cx <= x2 + 0.05 * w and y1 + 0.35 * h <= cy <= y2 + 0.10 * h:
                if area_min is None or w * h < area_min:
                    mejor, area_min = j, w * h
        if mejor is not None:
            vehiculos[mejor]['llantas'].append(i)
            l['vehiculo'] = mejor
    for v in vehiculos:
        v['ejes'], v['vista'], v['nota'] = analizar_ejes(v, [llantas[i] for i in v['llantas']])
    return vehiculos, llantas

def texto_ejes(v):
    return 'N/D' if v['ejes'] is None else str(v['ejes'])

def dibujar(img_rgb, vehiculos, llantas):
    out = img_rgb.copy()
    g = max(2, int(round(max(out.shape[:2]) / 450)))
    for v in vehiculos:
        col = COLORES[v['clase']]
        x1, y1, x2, y2 = map(int, v['box'])
        cv2.rectangle(out, (x1, y1), (x2, y2), col, g)
        texto = ETIQUETA_IMG[v['clase']] + ' | ' + str(len(v['llantas'])) + ' llantas | ejes: ' + texto_ejes(v)
        esc, gt = g * 0.33, max(1, g // 2)
        (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, esc, gt)
        cv2.rectangle(out, (x1, max(0, y1 - th - 8)), (x1 + tw + 6, y1), col, -1)
        cv2.putText(out, texto, (x1 + 3, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, esc, (255, 255, 255), gt, cv2.LINE_AA)
    for l in llantas:
        k = l['kps']
        x1, y1, x2, y2 = map(int, l['box'])
        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 255, 0), max(1, g // 2))
        for a, b in ESQUELETO:
            if k[a, 2] > UMBRAL_KP and k[b, 2] > UMBRAL_KP:
                cv2.line(out, tuple(map(int, k[a, :2])), tuple(map(int, k[b, :2])), (255, 255, 0), max(1, g // 2), cv2.LINE_AA)
        for i in range(len(k)):
            if k[i, 2] > UMBRAL_KP:
                cv2.circle(out, tuple(map(int, k[i, :2])), g + 2, KPT_COLORES[i], -1, cv2.LINE_AA)
                cv2.circle(out, tuple(map(int, k[i, :2])), g + 2, (0, 0, 0), 1, cv2.LINE_AA)
    return out

def tablas(vehiculos, llantas):
    tv = pd.DataFrame([{'Vehículo': j + 1, 'Tipo': CLASES[v['clase']], 'Confianza': round(v['conf'], 3),
                        'Llantas visibles': len(v['llantas']), 'Vista': v['vista'],
                        'Ejes estimados': texto_ejes(v), 'Nota': v['nota']}
                       for j, v in enumerate(vehiculos)])
    filas = []
    for i, l in enumerate(llantas):
        k = l['kps']
        filas.append({'Llanta': i + 1, 'Vehículo': (l['vehiculo'] + 1) if l['vehiculo'] is not None else '-',
                      'Confianza': round(l['conf'], 3),
                      'Centro (x, y)': (int(k[0, 0]), int(k[0, 1])), 'Contacto (x, y)': (int(k[1, 0]), int(k[1, 1])),
                      'Diámetro aprox. (px)': int(max(k[1, 1] - k[2, 1], k[4, 0] - k[3, 0]))})
    return tv, pd.DataFrame(filas)

def leyenda():
    for nombre, col in zip(KPT_NOMBRES, KPT_COLORES):
        plt.scatter([], [], color=np.array(col) / 255, edgecolors='k', label=nombre)
    plt.legend(loc='lower right', fontsize=8)

def ejecutar(nombre, img_bgr, conf, guardar=True):
    t0 = time.time()
    vehiculos, llantas = metodos_disponibles()[nombre](img_bgr, conf)
    dt = time.time() - t0
    vehiculos, llantas = asociar(vehiculos, llantas)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    vis = dibujar(img_rgb, vehiculos, llantas)
    plt.figure(figsize=(13, 8)); plt.imshow(vis); plt.title(nombre); plt.axis('off'); leyenda(); plt.show()
    print('Tiempo: ' + format(dt, '.2f') + ' s · vehículos: ' + str(len(vehiculos)) + ' · llantas: ' + str(len(llantas)))
    if not vehiculos or not llantas:
        print('Pocas o ninguna detección: baje la confianza (vehículos ~0.25, llantas ~0.10) o active DEPURAR para ver los puntajes crudos. '
              'Las miniaturas de baja resolución (p. ej. images.jpg de Google) dan llantas muy pequeñas.')
    tv, tl = tablas(vehiculos, llantas)
    display(tv); display(tl)
    if guardar:
        os.makedirs('/content/resultados', exist_ok=True)
        base = os.path.splitext(os.path.basename(IMG_PATH))[0]
        sufijo = nombre.split(' ')[0].lower()
        cv2.imwrite('/content/resultados/' + base + '_kp_' + sufijo + '.jpg', cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    return vehiculos, llantas

## 7. Interfaz: elegir imagen, método y umbral

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

def leer_fileupload(valor):
    items = valor.values() if isinstance(valor, dict) else valor     # ipywidgets 7 u 8
    res = []
    for it in items:
        nombre = it.get('name') or it.get('metadata', {}).get('name', 'subida.jpg')
        res.append((nombre, bytes(it['content'])))
    return res

subir = w.FileUpload(accept='image/*', multiple=True, description='Subir imagen')
selector_img = w.Dropdown(options=IMAGENES, value=IMG_PATH, description='Imagen:', layout=w.Layout(width='420px'))
selector_met = w.Dropdown(options=list(metodos_disponibles()), description='Método:', layout=w.Layout(width='520px'))
umbral = w.FloatSlider(value=0.25, min=0.05, max=0.90, step=0.05, description='Conf. vehículo:', style={'description_width': '110px'})
umbral_llanta = w.FloatSlider(value=0.15, min=0.02, max=0.90, step=0.01, description='Conf. llanta:', style={'description_width': '110px'})
depurar = w.Checkbox(value=False, description='Mostrar detecciones crudas')
boton = w.Button(description='Detectar keypoints', button_style='primary', icon='play')
salida = w.Output()

def al_subir(cambio):
    for nombre, contenido in leer_fileupload(subir.value):
        with open(nombre, 'wb') as f:
            f.write(contenido)
        if nombre not in IMAGENES:
            IMAGENES.append(nombre)
    selector_img.options = IMAGENES
    selector_img.value = IMAGENES[-1]

def al_pulsar(_):
    global IMG_PATH, CONF_LLANTA, DEPURAR
    import traceback
    IMG_PATH = selector_img.value
    CONF_LLANTA, DEPURAR = umbral_llanta.value, depurar.value
    boton.disabled = True
    with salida:
        clear_output(wait=True)
        print('Procesando ' + os.path.basename(IMG_PATH) + '... (la primera ejecución descarga los modelos)')
        try:
            img = cv2.imread(IMG_PATH)
            if img is None:
                raise FileNotFoundError('No se pudo leer la imagen: ' + IMG_PATH)
            clear_output(wait=True)
            ejecutar(selector_met.value, img, umbral.value)
        except Exception:
            traceback.print_exc()
        finally:
            boton.disabled = False

subir.observe(al_subir, names='value')
boton.on_click(al_pulsar)
display(w.VBox([w.HBox([subir, selector_img]), selector_met, w.HBox([umbral, umbral_llanta]), depurar, boton, salida]))

## 8. Entrenamiento de modelos de keypoints

### 8.1 Imágenes para el dataset
Suba un **.zip** con fotos de vehículos (idealmente vistas laterales, 200 o más). Si ya tiene un dataset de keypoints etiquetado en Roboflow, puede saltar a 8.3 y apuntar `DATA_YAML` a su `data.yaml` (mismo orden de keypoints).

In [ ]:
DIR_IMGS = '/content/imagenes'
shutil.rmtree(DIR_IMGS, ignore_errors=True)
os.makedirs(DIR_IMGS, exist_ok=True)

for nombre in files.upload().keys():
    if nombre.lower().endswith('.zip'):
        !unzip -qo "$nombre" -d /content/imagenes
    else:
        shutil.move(nombre, DIR_IMGS)

EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
LISTA_IMGS = sorted(
    p for p in glob.glob(DIR_IMGS + '/**/*', recursive=True)
    if p.lower().endswith(EXT) and not os.path.basename(p).startswith('.')
)
print(f"Total imágenes cargadas: {len(LISTA_IMGS)}")

Saving muestra_m3.zip to muestra_m3.zip
Total imágenes cargadas: 250


### 8.2 Auto-etiquetado en formato YOLO-pose
Genera `images/` y `labels/` con split train/val. Revise y corrija las etiquetas en Roboflow (proyecto de tipo *Keypoint Detection*) antes de entrenar para obtener mejores resultados: un modelo entrenado solo con etiquetas automáticas reproduce los errores del método zero-shot.

In [ ]:
DIR_DS = '/content/dataset_llantas'
CONF_LLANTA = 0.14          # Umbral óptimo para que YOLOE zero-shot capture todas las llantas
CONF_ETIQUETADO = 0.14
PORC_VAL = 0.2

def linea_yolo_pose(l, W, H):
    x1, y1, x2, y2 = l['box']
    k = l['kps']
    x1, y1 = min(x1, k[:, 0].min()), min(y1, k[:, 1].min())
    x2, y2 = max(x2, k[:, 0].max()), max(y2, k[:, 1].max())
    vals = [(x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H]
    for x, y, _ in k:
        vals += [min(max(x / W, 0), 1), min(max(y / H, 0), 1), 2]
    return '0 ' + ' '.join(format(v, '.6f') if isinstance(v, float) else str(v) for v in vals)

# Filtrar solo imágenes donde sí se detecten llantas (0 fondos vacíos)
etiquetadas = []
for ruta in LISTA_IMGS:
    im = cv2.imread(ruta)
    if im is None:
        continue
    _, llantas = kp_zeroshot(im, CONF_ETIQUETADO)
    if llantas:
        etiquetadas.append((ruta, im, llantas))

shutil.rmtree(DIR_DS, ignore_errors=True)
for s in ('train', 'val'):
    os.makedirs(DIR_DS + '/images/' + s)
    os.makedirs(DIR_DS + '/labels/' + s)

random.seed(0)
random.shuffle(etiquetadas)
n_val = max(1, int(len(etiquetadas) * PORC_VAL))
total_llantas = 0

for i, (ruta, im, llantas) in enumerate(etiquetadas):
    split = 'val' if i < n_val else 'train'
    H, W = im.shape[:2]
    base = str(i).zfill(5) + '_' + os.path.splitext(os.path.basename(ruta))[0]
    cv2.imwrite(DIR_DS + '/images/' + split + '/' + base + '.jpg', im)
    with open(DIR_DS + '/labels/' + split + '/' + base + '.txt', 'w') as f:
        f.write('\n'.join(linea_yolo_pose(l, W, H) for l in llantas))
    total_llantas += len(llantas)

DATA_YAML = DIR_DS + '/data.yaml'
with open(DATA_YAML, 'w') as f:
    f.write('path: ' + DIR_DS + '\ntrain: images/train\nval: images/val\n')
    f.write('kpt_shape: [5, 3]\nflip_idx: ' + str(FLIP_IDX) + '\n')
    f.write('names:\n  0: llanta\n')

print(f'Imágenes válidas: {len(etiquetadas)} | Total llantas etiquetadas: {total_llantas}')

Imágenes válidas: 73 | Total llantas etiquetadas: 105


### 8.3 Entrenar YOLO-pose

In [ ]:
from google.colab import drive

modelo_pose = YOLO('yolo11s-pose.pt')
modelo_pose.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=15,
    pose=12.0,
    close_mosaic=10,
    device=0,
    project='/content/runs_pose',
    name='m3_ejes',
    exist_ok=True,
)

print('mAP50 keypoints:', YOLO(RUTA_YOLOPOSE).val(data=DATA_YAML).pose.map50)

# Guardar respaldo en Google Drive y descargar para FastAPI
shutil.copy(RUTA_YOLOPOSE, '/content/m3_ejes.pt')
drive.mount('/content/drive')
shutil.copy(RUTA_YOLOPOSE, '/content/drive/MyDrive/m3_ejes.pt')
files.download('/content/m3_ejes.pt')

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_llantas/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=m3_ejes, nbs=64, nms=None, opset=None, optimize=F

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 8.4 Entrenar Detectron2 Keypoint R-CNN
Se convierten las etiquetas YOLO-pose a formato COCO y se ajusta el modelo `keypoint_rcnn_R_50_FPN_3x`.

In [ ]:
def yolo_a_coco(split):
    dir_img = DIR_DS + '/images/' + split
    images, anns = [], []
    for idx, ruta in enumerate(sorted(glob.glob(dir_img + '/*.jpg'))):
        H, W = cv2.imread(ruta).shape[:2]
        images.append({'id': idx, 'file_name': os.path.basename(ruta), 'width': W, 'height': H})
        lbl = ruta.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if not os.path.exists(lbl):
            continue
        for linea in open(lbl).read().split('\n'):
            v = linea.split()
            if len(v) < 5 + 3 * len(KPT_NOMBRES):
                continue
            cx, cy, bw, bh = [float(t) for t in v[1:5]]
            box = [(cx - bw / 2) * W, (cy - bh / 2) * H, bw * W, bh * H]
            kps = []
            for j in range(len(KPT_NOMBRES)):
                x, y, vis = float(v[5 + 3 * j]), float(v[6 + 3 * j]), int(float(v[7 + 3 * j]))
                kps += [x * W, y * H, 2 if vis > 0 else 0]
            anns.append({'id': len(anns) + 1, 'image_id': idx, 'category_id': 1, 'bbox': box,
                         'area': box[2] * box[3], 'iscrowd': 0, 'keypoints': kps,
                         'num_keypoints': sum(1 for t in kps[2::3] if t > 0)})
    coco = {'images': images, 'annotations': anns,
            'categories': [{'id': 1, 'name': 'llanta', 'keypoints': KPT_NOMBRES,
                            'skeleton': [[a + 1, b + 1] for a, b in ESQUELETO]}]}
    ruta_json = DIR_DS + '/coco_' + split + '.json'
    json.dump(coco, open(ruta_json, 'w'))
    return ruta_json, dir_img

for split in ('train', 'val'):
    nombre = 'llantas_' + split
    if nombre in DatasetCatalog.list():
        DatasetCatalog.remove(nombre); MetadataCatalog.remove(nombre)
    ruta_json, dir_img = yolo_a_coco(split)
    register_coco_instances(nombre, {}, ruta_json, dir_img)
    MetadataCatalog.get(nombre).set(keypoint_names=KPT_NOMBRES,
                                    keypoint_flip_map=[('izquierda', 'derecha')],
                                    keypoint_connection_rules=[(KPT_NOMBRES[a], KPT_NOMBRES[b], (255, 255, 0)) for a, b in ESQUELETO])

cfg = construir_cfg_d2()
cfg.DATASETS.TRAIN = ('llantas_train',)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.0025
cfg.SOLVER.MAX_ITER = 3000
cfg.SOLVER.STEPS = (2000, 2600)
cfg.SOLVER.CHECKPOINT_PERIOD = 1000
os.makedirs(DIR_D2, exist_ok=True)
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)     # el cabezal de 17 keypoints (humano) se reinicia a 5
trainer.train()

In [ ]:
# Evaluación COCO (AP de keypoints con OKS) sobre validación
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
cfg_eval = construir_cfg_d2()
cfg_eval.MODEL.WEIGHTS = RUTA_D2
cfg_eval.DATASETS.TEST = ('llantas_val',)
pred_eval = DefaultPredictor(cfg_eval)
evaluador = COCOEvaluator('llantas_val', output_dir=DIR_D2 + '/eval')
print(inference_on_dataset(pred_eval.model, build_detection_test_loader(cfg_eval, 'llantas_val'), evaluador))

> Tras entrenar, vuelva a ejecutar la **celda 7** para que los nuevos métodos aparezcan en la interfaz. Los pesos se borran al cerrar Colab: copie `/content/runs` a Google Drive si desea conservarlos.

## 9. Comparación de métodos sobre la misma imagen

In [ ]:
CONF = 0.25
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
met = metodos_disponibles()
fig, ax = plt.subplots(1, len(met), figsize=(9 * len(met), 6), squeeze=False)
resumen = []
for a, (nombre, fn) in zip(ax[0], met.items()):
    t0 = time.time()
    v, l = asociar(*fn(img, CONF))
    resumen.append({'Método': nombre, 'Vehículos': len(v), 'Llantas': len(l),
                    'Ejes (total)': sum(x['ejes'] or 0 for x in v), 'Tiempo (s)': round(time.time() - t0, 2)})
    a.imshow(dibujar(img_rgb, v, l)); a.set_title(nombre, fontsize=10); a.axis('off')
plt.tight_layout(); plt.show()
display(pd.DataFrame(resumen).set_index('Método'))

## 10. Descargar resultados

In [ ]:
!cd /content && zip -qr resultados_keypoints.zip resultados && ls resultados
files.download('/content/resultados_keypoints.zip')


zip error: Nothing to do! (try: zip -qr resultados_keypoints.zip . -i resultados)


FileNotFoundError: Cannot find file: /content/resultados_keypoints.zip